In [1]:
import numpy as np
import h5py
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torchdiffeq import odeint

from sklearn.decomposition import PCA
from time import time



## Dataset Inspection

In [2]:
h5_path = r"C:\Users\arnab\OneDrive\Desktop\Study material\Research\Tear Film PDE\datasets\pde_dataset_short.h5"

with h5py.File(h5_path, "r+") as f:
    print("Keys in file:", list(f.keys()))
    for key in f.keys():
        dset = f[key]
        print(f"{key}: shape={dset.shape}, dtype={dset.dtype}")
        # Calculate memory required in bytes
        mem_required = dset.size * dset.dtype.itemsize
        print(f"Memory required to load '{key}': {mem_required} bytes ({mem_required / (1024 ** 2):.2f} MB)")
    para_name = f["para_name"][:]
    para = f["para"][:]
    print("para_name data:", para_name)
    
    # I_data = f["I"][:]  # shape (16068, 81, 601)
    # I_data_reshaped = np.transpose(I_data, (0, 2, 1))  # (16068, 601, 81)
    # del f["I"]  # remove the original dataset
    # f.create_dataset("I", data=I_data_reshaped, dtype=I_data.dtype)
    # print("Reshaped 'I' dataset to", f["I"].shape)
    # para_reshaped = np.reshape(para, (16068, 6))
    # del f["para"]
    # f.create_dataset("para", data=para_reshaped, dtype=para_reshaped.dtype)
    # print("Reshaped 'para' dataset to", f["para"].shape)



Keys in file: ['I', 'c', 'f', 'h', 'para', 'para_name']
I: shape=(16068, 101, 81), dtype=float32
Memory required to load 'I': 525809232 bytes (501.45 MB)
c: shape=(16068, 101, 81), dtype=float32
Memory required to load 'c': 525809232 bytes (501.45 MB)
f: shape=(16068, 101, 81), dtype=float32
Memory required to load 'f': 525809232 bytes (501.45 MB)
h: shape=(16068, 101, 81), dtype=float32
Memory required to load 'h': 525809232 bytes (501.45 MB)
para: shape=(16068, 6), dtype=float64
Memory required to load 'para': 771264 bytes (0.74 MB)
para_name: shape=(6,), dtype=object
Memory required to load 'para_name': 48 bytes (0.00 MB)
para_name data: [b't_s' b'd' b'f_initial' b'd_sigma_0' b'R_I' b'v']


In [3]:
# with h5py.File(h5_path, "r+") as f:
#     p_data = f["p"][:]  # shape (81, 601, 16068)
#     p_data_reshaped = np.transpose(p_data, (2, 1, 0))  # (16068, 81, 601)
#     del f["p"]  # remove the original dataset
#     f.create_dataset("p", data=p_data_reshaped, dtype=p_data.dtype)
#     print("Reshaped 'p' dataset to", f["p"].shape)


## Dataset management and loading

In [4]:
class TearFilmH5Dataset(Dataset):
    def __init__(self, h5_path, indices=None, dtype=torch.float32):
        """
        h5_path : path to filtered_data_1.h5
        indices : optional array of case indices (for train/test split)
        dtype   : dtype to cast floats to (use float32 for efficiency)
        """
        self.h5_path = h5_path
        self._file = None
        self.dtype = dtype

        # Read global length from file once (main process only)
        with h5py.File(self.h5_path, "r") as f:
            N = f["I"].shape[0]
        if indices is None:
            self.indices = np.arange(N, dtype=np.int64)
        else:
            self.indices = np.asarray(indices, dtype=np.int64)

    def _get_file(self):
        # Lazily open a private file handle (important for num_workers > 0)
        if self._file is None:
            self._file = h5py.File(self.h5_path, "r")
        return self._file

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        f = self._get_file()
        i = int(self.indices[idx])

        # Read from HDF5 (each is a NumPy array slice; only one sample is loaded)
        I  = f["I"][i]      # (601, 81), float64
        h  = f["h"][i]      # (601, 81), float64
        c  = f["c"][i]      # (601, 81), float64
        ff  = f["f"][i]      # (601, 81), float64
        p  = f["para"][i]   # (6,),      float64

        # Convert to float32 tensors, rearrange to channels-first
        I  = torch.as_tensor(I,  dtype=self.dtype).unsqueeze(0)   # (1, 601, 81)
        h  = torch.as_tensor(h,  dtype=self.dtype).unsqueeze(0)   # (1, 601, 81)
        c  = torch.as_tensor(c,  dtype=self.dtype).unsqueeze(0)   # (1, 601, 81)
        ff  = torch.as_tensor(ff,  dtype=self.dtype).unsqueeze(0)   # (1, 601, 81)
        p  = torch.as_tensor(p,  dtype=self.dtype)                # (6,)

        # Stack h and c along channel dimension as one target
        y = torch.cat([h, c, ff], dim=0)  # (3, 601, 81)

        return (I, p), y


In [5]:
def make_train_test_datasets(
    h5_path,
    subset_frac=1.0,
    train_frac=0.8,
    seed=42,
    dtype=torch.float32,
):
    """
    h5_path     : path to .h5 file
    subset_frac : fraction of the full dataset to use overall (0 < subset_frac <= 1)
    train_frac  : fraction of the subset used for training (0 < train_frac < 1)
    seed        : RNG seed for both subset selection and train/test split
    dtype       : dtype passed to TearFilmH5Dataset
    """
    if not (0.0 < subset_frac <= 1.0):
        raise ValueError("subset_frac must be in (0,1].")
    if not (0.0 < train_frac < 1.0):
        raise ValueError("train_frac must be in (0,1).")

    # Read total number of samples
    with h5py.File(h5_path, "r") as f:
        N = f["I"].shape[0]

    rng = np.random.RandomState(seed)
    perm_all = rng.permutation(N)

    # Choose the global subset
    subset_size   = int(round(subset_frac * N))
    subset_indices = perm_all[:subset_size]

    # Split the subset into train and test
    n_train = int(round(train_frac * subset_size))
    train_indices = subset_indices[:n_train]
    test_indices  = subset_indices[n_train:]

    # Build datasets
    train_dataset = TearFilmH5Dataset(
        h5_path,
        indices=train_indices,
        dtype=dtype,
    )
    test_dataset = TearFilmH5Dataset(
        h5_path,
        indices=test_indices,
        dtype=dtype,
    )
    return train_dataset, test_dataset


In [6]:
train_dataset, test_dataset = make_train_test_datasets(
    h5_path,
    subset_frac=0.01,   # use 100% of the data
    train_frac=0.75,    # 80% of that subset is training
    seed=42,
    dtype=torch.float32,
)

In [7]:
print("len(train_dataset) =", len(train_dataset))
print("len(test_dataset)  =", len(test_dataset))

# Try accessing a few samples directly
(I0, p0), y0 = train_dataset[0]
print("I0.shape:", I0.shape)  # expect (1, 601, 81)
print("p0.shape:", p0.shape)  # expect (6,)
print("y0.shape:", y0.shape)  # expect (2, 601, 81)

len(train_dataset) = 121
len(test_dataset)  = 40
I0.shape: torch.Size([1, 101, 81])
p0.shape: torch.Size([6])
y0.shape: torch.Size([3, 101, 81])


In [8]:
batch_size = 32  # or whatever you like

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,        # <- IMPORTANT
    pin_memory=True,      # if using CUDA
    # persistent_workers=False by default
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,        # <- IMPORTANT
    pin_memory=True,
)


# PCA Analysis

In [9]:
train_indices = train_dataset.indices  # (n_train,)
test_indices  = test_dataset.indices   # (n_test,)

with h5py.File(h5_path, "r") as f:
    # Example: use the 'c' dataset; shape should be (N_cases, T, N_r) = (16068, 101, 81)
    c_all = f["I"][:]

# Split into train/test using the same indices as your PyTorch datasets
c_train = c_all[train_indices]  # (n_train, T, N_r)
c_test  = c_all[test_indices]   # (n_test,  T, N_r)

# Reshape so that PCA sees spatial dimension only
# Shape: (n_train*T, N_r)
n_train, T, N_r = c_train.shape
X_train = c_train.reshape(n_train * T, N_r)

n_test = c_test.shape[0]
X_test  = c_test.reshape(n_test * T, N_r)

# Mean-center (PCA will do this internally, but we will keep the mean for reconstruction)
# You can let sklearn handle centering; I'll show the standard usage.


In [10]:
# Fit PCA: n_components can be up to N_r, but we only need the spectrum initially
pca_full = PCA(n_components=N_r)  # N_r = 81
pca_full.fit(X_train)

explained_var_ratio = np.cumsum(pca_full.explained_variance_ratio_)

In [11]:
for target in [0.99, 0.999, 0.9999]:
    d = int(np.searchsorted(explained_var_ratio, target) + 1)
    print(f"{target*100:.3f}% variance -> d = {d}")

99.000% variance -> d = 2
99.900% variance -> d = 4
99.990% variance -> d = 7


In [12]:
def pca_reconstruction_error(pca_model, X_test, d):
    """
    pca_model: fitted PCA with n_components = N_r
    X_train, X_test: (M, N_r)
    d: number of principal components to keep
    """
    # Project onto first d components
    # sklearn's PCA doesn't have a direct 'partial inverse' for smaller d,
    # but we can construct it using components_ and mean_
    U_d = pca_model.components_[:d, :]          # (d, N_r)
    mean = pca_model.mean_                      # (N_r,)

    # Scores on test set
    # scores = (X - mean) @ U_d^T
    Xc_test = X_test - mean
    scores_test = Xc_test @ U_d.T               # (M_test, d)

    # Reconstruction: mean + scores @ U_d
    X_hat_test = mean + scores_test @ U_d       # (M_test, N_r)

    # Relative MSE
    num = np.mean(np.sum((X_test - X_hat_test)**2, axis=1))
    den = np.mean(np.sum(X_test**2, axis=1)) + 1e-12
    rel_mse = num / den
    return rel_mse

# Example: evaluate a range of d
candidate_ds = [2, 4, 6, 8, 10, 12, 16, 20, 24, 32]
for d in candidate_ds:
    err = pca_reconstruction_error(pca_full, X_test, d)
    print(f"d = {d:2d}, relative reconstruction MSE (test) = {err}")


d =  2, relative reconstruction MSE (test) = 0.0007793557015247643
d =  4, relative reconstruction MSE (test) = 0.000110698449134361
d =  6, relative reconstruction MSE (test) = 1.454823632229818e-05
d =  8, relative reconstruction MSE (test) = 3.3369610719091725e-06
d = 10, relative reconstruction MSE (test) = 8.228811338995001e-07
d = 12, relative reconstruction MSE (test) = 2.519049360216741e-07
d = 16, relative reconstruction MSE (test) = 1.4296827544058033e-07
d = 20, relative reconstruction MSE (test) = 1.2107616953471734e-07
d = 24, relative reconstruction MSE (test) = 1.1619005846341679e-07
d = 32, relative reconstruction MSE (test) = 9.742479534224913e-08


d=8 seems to be a good choice for the number of dimensions to reduce to. The explained variance ratio indicates that the first 8 principal components capture a significant portion of the variance in the dataset, making it a suitable choice for dimensionality reduction while retaining important information.

# Models

In [13]:
class Encoder(nn.Module):
    """
    DeepONet-style encoder: z = E(U, lam) = (T(lam))^T b(U)

    Inputs:
      - U:   (B, 3, N)  snapshot with channels [h, c, f] over N spatial points
      - lam: (B, 6)     parameter vector [t_s, h0, f0, d_sigma_0, RI, v]

    Output:
      - z:   (B, d)     latent code

    Notes:
      - b(U) is a 1D-CNN branch producing features in R^p
      - T(lam) is an MLP producing a matrix in R^{p x d}
      - Merge is bilinear: z_j = <b(U), T_j(lam)>
    """
    def __init__(
        self,
        N: int = 81,
        lam_dim: int = 6,
        latent_dim: int = 8,     # d
        feature_dim: int = 256,   # p
        conv_channels=(32, 64, 128),
        kernel_size: int = 5,
        use_groupnorm: bool = False,
        gn_groups: int = 8,
        mlp_hidden=(128, 256),
        activation: str = "gelu",
        dropout: float = 0.0,
    ):
        super().__init__()
        self.N = N
        self.lam_dim = lam_dim
        self.latent_dim = latent_dim
        self.feature_dim = feature_dim

        act = nn.GELU if activation.lower() == "gelu" else nn.ReLU

        # ---- State branch b(U): Conv1D stack -> GlobalAvgPool -> MLP -> R^p ----
        c0 = 3
        convs = []
        in_ch = c0
        for out_ch in conv_channels:
            convs.append(nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, stride=2, padding=kernel_size // 2))
            if use_groupnorm:
                convs.append(nn.GroupNorm(num_groups=min(gn_groups, out_ch), num_channels=out_ch))
            convs.append(act())
            if dropout > 0:
                convs.append(nn.Dropout(dropout))
            in_ch = out_ch
        self.state_conv = nn.Sequential(*convs)

        # After convs, we'll global-average pool over the (reduced) spatial dimension -> (B, in_ch)
        state_mlp_layers = []
        prev = in_ch
        for h in mlp_hidden:
            state_mlp_layers.append(nn.Linear(prev, h))
            state_mlp_layers.append(act())
            if dropout > 0:
                state_mlp_layers.append(nn.Dropout(dropout))
            prev = h
        state_mlp_layers.append(nn.Linear(prev, feature_dim))
        self.state_mlp = nn.Sequential(*state_mlp_layers)

        # ---- Parameter branch T(lam): MLP -> R^{p*d} reshape to (B, p, d) ----
        param_mlp_layers = []
        prev = lam_dim
        for h in mlp_hidden:
            param_mlp_layers.append(nn.Linear(prev, h))
            param_mlp_layers.append(act())
            if dropout > 0:
                param_mlp_layers.append(nn.Dropout(dropout))
            prev = h
        param_mlp_layers.append(nn.Linear(prev, feature_dim * latent_dim))
        self.param_mlp = nn.Sequential(*param_mlp_layers)

        # Optional: stabilize dot-products by normalizing branch outputs
        self.normalize_branches = True

        # Small epsilon for normalization
        self.eps = 1e-8

    def forward(self, U: torch.Tensor, lam: torch.Tensor) -> torch.Tensor:
        """
        Args:
          U:   (B, 3, N)
          lam: (B, 6)

        Returns:
          z:   (B, d)
        """
        if U.dim() != 3:
            raise ValueError(f"U must have shape (B, 3, N). Got {tuple(U.shape)}")
        if U.size(1) != 3:
            raise ValueError(f"U must have 3 channels [h,c,f]. Got {U.size(1)}")
        if lam.dim() != 2 or lam.size(1) != self.lam_dim:
            raise ValueError(f"lam must have shape (B, {self.lam_dim}). Got {tuple(lam.shape)}")

        # b(U): (B, 3, N) -> (B, C_last, L) -> GAP -> (B, C_last) -> (B, p)
        x = self.state_conv(U)
        x = x.mean(dim=-1)  # global average pooling over spatial dim
        b = self.state_mlp(x)  # (B, p)

        # T(lam): (B, 6) -> (B, p*d) -> (B, p, d)
        T_flat = self.param_mlp(lam)  # (B, p*d)
        T = T_flat.view(lam.size(0), self.feature_dim, self.latent_dim)  # (B, p, d)

        if self.normalize_branches:
            # Normalize b to unit norm per batch item
            b = b / (b.norm(dim=1, keepdim=True) + self.eps)  # (B, p)
            # Normalize columns of T (each latent direction) to unit norm per batch item
            T = T / (T.norm(dim=1, keepdim=True) + self.eps)  # (B, p, d)

        # Bilinear merge: z = T^T b  -> (B, d)
        # Compute batch-wise: z[b, j] = sum_p T[b, p, j] * b[b, p]
        z = torch.einsum("bp,bpd->bd", b, T)

        return z


In [14]:
class FiLM(nn.Module):
    """
    FiLM generator: (gamma, beta) = FiLM(lam) for a given hidden width.
    Produces per-feature affine modulation.
    """
    def __init__(self, lam_dim: int, width: int, hidden=(64, 128), activation="gelu", dropout: float = 0.0):
        super().__init__()
        act = nn.GELU if activation.lower() == "gelu" else nn.ReLU

        layers = []
        prev = lam_dim
        for h in hidden:
            layers.append(nn.Linear(prev, h))
            layers.append(act())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 2 * width))  # gamma and beta
        self.net = nn.Sequential(*layers)

    def forward(self, lam: torch.Tensor):
        """
        lam: (B, lam_dim)
        returns gamma, beta: each (B, width)
        """
        gb = self.net(lam)
        gamma, beta = torch.chunk(gb, 2, dim=-1)
        return gamma, beta


class LatentDynamics(nn.Module):
    """
    FiLM-conditioned residual MLP for latent ODE dynamics:
        dz/dt = F_phi(z, lam)

    Designed to work with torchdiffeq.odeint:
        dynamics(t, z, lam) -> dzdt

    Inputs:
      - t: scalar tensor (ignored by default; autonomous)
      - z: (B, d)
      - lam: (B, 6)

    Output:
      - dzdt: (B, d)
    """
    def __init__(
        self,
        latent_dim: int = 16,    # d
        lam_dim: int = 6,
        width: int = 128,
        depth: int = 4,          # number of residual blocks
        film_hidden=(64, 128),
        activation: str = "gelu",
        dropout: float = 0.0,
        use_layernorm: bool = True,
        autonomous: bool = True,  # if False, t is concatenated as an input feature
    ):
        super().__init__()
        self.latent_dim = latent_dim
        self.lam_dim = lam_dim
        self.width = width
        self.depth = depth
        self.autonomous = autonomous

        act = nn.GELU if activation.lower() == "gelu" else nn.ReLU

        in_dim = latent_dim if autonomous else latent_dim + 1

        # Lift to hidden width
        self.in_proj = nn.Linear(in_dim, width)

        # Residual blocks
        self.blocks = nn.ModuleList()
        self.norms = nn.ModuleList() if use_layernorm else None
        self.films = nn.ModuleList()

        for _ in range(depth):
            block = nn.Sequential(
                nn.Linear(width, width),
                act(),
                nn.Dropout(dropout) if dropout > 0 else nn.Identity(),
                nn.Linear(width, width),
            )
            self.blocks.append(block)
            if use_layernorm:
                self.norms.append(nn.LayerNorm(width))
            self.films.append(FiLM(lam_dim=lam_dim, width=width, hidden=film_hidden, activation=activation, dropout=dropout))

        # Project back to latent dimension
        self.out_proj = nn.Linear(width, latent_dim)

        # Optional: small scaling of output to improve stability at init
        self.output_scale = nn.Parameter(torch.tensor(1.0))

    def forward(self, t: torch.Tensor, z: torch.Tensor, lam: torch.Tensor) -> torch.Tensor:
        """
        t:   scalar tensor or (B,) tensor (ignored if autonomous=True)
        z:   (B, d)
        lam: (B, lam_dim)
        """
        if z.dim() != 2 or z.size(1) != self.latent_dim:
            raise ValueError(f"z must have shape (B, {self.latent_dim}). Got {tuple(z.shape)}")
        if lam.dim() != 2 or lam.size(1) != self.lam_dim:
            raise ValueError(f"lam must have shape (B, {self.lam_dim}). Got {tuple(lam.shape)}")

        if self.autonomous:
            x = z
        else:
            # Ensure t is broadcastable to (B,1)
            if t.dim() == 0:
                t_in = t.expand(z.size(0), 1)
            elif t.dim() == 1:
                t_in = t.view(-1, 1)
                if t_in.size(0) == 1:
                    t_in = t_in.expand(z.size(0), 1)
                elif t_in.size(0) != z.size(0):
                    raise ValueError(f"t has batch {t_in.size(0)} but z has batch {z.size(0)}")
            else:
                raise ValueError(f"t must be scalar or 1D. Got {tuple(t.shape)}")
            x = torch.cat([z, t_in], dim=1)  # (B, d+1)

        h = self.in_proj(x)  # (B, width)

        for i in range(self.depth):
            # FiLM modulation
            gamma, beta = self.films[i](lam)  # (B,width), (B,width)

            # Residual block
            r = self.blocks[i](h)  # (B,width)
            # Apply FiLM to the residual (common choice)
            r = gamma * r + beta

            h = h + r  # residual connection
            if self.norms is not None:
                h = self.norms[i](h)

        dzdt = self.out_proj(h) * self.output_scale  # (B, d)
        return dzdt


class ODEFuncWrapper(nn.Module):
    """
    Wrap LatentDynamics so it matches torchdiffeq signature f(t, z) -> dzdt
    while holding lam fixed for the batch.
    """
    def __init__(self, dynamics: LatentDynamics):
        super().__init__()
        self.dynamics = dynamics
        self._lam = None

    def set_lam(self, lam: torch.Tensor):
        self._lam = lam

    def forward(self, t: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
        if self._lam is None:
            raise RuntimeError("lam is not set. Call set_lam(lam) before integrating.")
        return self.dynamics(t, z, self._lam)


In [15]:
class Decoder(nn.Module):
    """
    Dense -> reshape -> (upsample + Conv1D)* -> output 3 channels.

    Maps latent z(t) in R^d to a single spatial snapshot U_hat = (h_hat, c_hat, f_hat) in R^{3 x N}.

    Input:
      - z: (B, d)

    Output:
      - U_hat: (B, 3, N)

    Notes:
      - Uses explicit upsampling to hit N=81 exactly (since 81 is not a power of 2).
      - No positivity constraints are applied here (to be added later if desired).
    """
    def __init__(
        self,
        N: int = 81,
        latent_dim: int = 16,        # d
        base_channels: int = 64,      # C0
        base_length: int = 9,         # L0; choose so that you can upsample to 81 exactly
        up_lengths=(27, 81),          # explicit spatial lengths after each upsample
        conv_channels=(64, 32),       # channels after each conv stage
        kernel_size: int = 5,
        mlp_hidden=(128, 256),
        activation: str = "gelu",
        dropout: float = 0.0,
        use_groupnorm: bool = False,
        gn_groups: int = 8,
    ):
        super().__init__()
        self.N = N
        self.latent_dim = latent_dim
        self.base_channels = base_channels
        self.base_length = base_length

        if up_lengths[-1] != N:
            raise ValueError(f"up_lengths must end with N={N}. Got up_lengths={up_lengths}")
        if len(up_lengths) != len(conv_channels):
            raise ValueError("len(up_lengths) must equal len(conv_channels)")

        act = nn.GELU if activation.lower() == "gelu" else nn.ReLU

        # ---- Dense lift: z -> (C0 * L0) ----
        layers = []
        prev = latent_dim
        for h in mlp_hidden:
            layers.append(nn.Linear(prev, h))
            layers.append(act())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, base_channels * base_length))
        self.mlp = nn.Sequential(*layers)

        # ---- Upsample + Conv refinement blocks ----
        blocks = []
        in_ch = base_channels
        for out_ch, L_out in zip(conv_channels, up_lengths):
            block_layers = []

            # explicit resize to L_out
            block_layers.append(nn.Upsample(size=L_out, mode="linear", align_corners=False))

            # conv refine
            block_layers.append(nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding=kernel_size // 2))
            if use_groupnorm:
                block_layers.append(nn.GroupNorm(num_groups=min(gn_groups, out_ch), num_channels=out_ch))
            block_layers.append(act())
            if dropout > 0:
                block_layers.append(nn.Dropout(dropout))

            # second conv refine (optional but helps)
            block_layers.append(nn.Conv1d(out_ch, out_ch, kernel_size=kernel_size, padding=kernel_size // 2))
            if use_groupnorm:
                block_layers.append(nn.GroupNorm(num_groups=min(gn_groups, out_ch), num_channels=out_ch))
            block_layers.append(act())
            if dropout > 0:
                block_layers.append(nn.Dropout(dropout))

            blocks.append(nn.Sequential(*block_layers))
            in_ch = out_ch

        self.blocks = nn.ModuleList(blocks)

        # ---- Final projection to 3 channels ----
        self.out_conv = nn.Conv1d(in_ch, 3, kernel_size=1)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        """
        z: (B, d) -> U_hat: (B, 3, N)
        """
        if z.dim() != 2 or z.size(1) != self.latent_dim:
            raise ValueError(f"z must have shape (B, {self.latent_dim}). Got {tuple(z.shape)}")

        B = z.size(0)

        x = self.mlp(z)  # (B, C0*L0)
        x = x.view(B, self.base_channels, self.base_length)  # (B, C0, L0)

        for blk in self.blocks:
            x = blk(x)  # progressively to lengths in up_lengths

        U_hat = self.out_conv(x)  # (B, 3, N)
        if U_hat.size(-1) != self.N:
            raise RuntimeError(f"Decoder produced length {U_hat.size(-1)} but expected N={self.N}")

        return U_hat


In [16]:
class LatentODESurrogate(nn.Module):
    """
    Wrapper model that binds:
      - encoder:  z0 = E(U0, lam)
      - dynamics: dz/dt = F(z, lam)
      - decoder:  U_hat(t) = D(z(t))

    Forward input:
      - lam: (B, 6)

    Forward outputs (always):
      - U_hat: (B, T, 3, N)  where channels are [h, c, f]

    Optional outputs (if return_I=True):
      - I_hat: (B, T, N)  computed from predicted h,f and normalized so I(0, r0)=1
    """
    def __init__(
        self,
        encoder: nn.Module,
        dynamics: nn.Module,  # should support dynamics(t, z, lam) -> dzdt
        decoder: nn.Module,
        N: int = 81,
        lam_dim: int = 6,
        phi: float = 1.0,     # extinction coefficient (nondimensional)
        r0_index: int = 0,    # index for r=0 location in your grid (adjust if needed)
        eps: float = 1e-8,
    ):
        super().__init__()
        self.encoder = encoder
        self.dynamics = dynamics
        self.decoder = decoder
        self.N = N
        self.lam_dim = lam_dim
        self.phi = float(phi)
        self.r0_index = int(r0_index)
        self.eps = float(eps)

    def build_initial_state(self, lam: torch.Tensor, dtype=None) -> torch.Tensor:
        """
        Construct U0 = [1_N, 1_N, f0*1_N]^T from lam.
        Assumes lam = [t_s, h0, f0, d_sigma0, RI, v].
        """
        if dtype is None:
            dtype = lam.dtype
        B = lam.size(0)
        device = lam.device
        ones = torch.ones((B, 1, self.N), device=device, dtype=dtype)
        f0 = lam[:, 2].view(B, 1, 1).to(dtype)  # f0 component
        U0 = torch.cat([ones, ones, f0 * ones], dim=1)  # (B, 3, N)
        return U0

    def compute_intensity(self, U_hat: torch.Tensor) -> torch.Tensor:
        """
        Compute I_hat from predicted h,f and enforce I(0, r0)=1 via learned I0 per sample.
        U_hat: (B, T, 3, N)
        Returns I_hat: (B, T, N)
        """
        h_hat = U_hat[:, :, 0, :]  # (B,T,N)
        f_hat = U_hat[:, :, 2, :]  # (B,T,N)

        # Unnormalized intensity (without I0)
        numer = 1.0 - torch.exp(-self.phi * f_hat * h_hat)
        denom = 1.0 + f_hat**2
        I_raw = numer / (denom + self.eps)  # (B,T,N)

        # Compute I0 per sample from predicted values at (t=0, r=r0_index)
        r0 = self.r0_index
        I_raw_00 = I_raw[:, 0, r0]  # (B,)
        I0 = 1.0 / (I_raw_00 + self.eps)  # (B,)
        I_hat = I_raw * I0.view(-1, 1, 1)  # broadcast (B,1,1)

        return I_hat

    def forward(
        self,
        lam: torch.Tensor,
        t: torch.Tensor = None,
        method: str = "dopri5",
        return_I: bool = False,
        return_latent: bool = False,
        ode_options: dict = None,
    ):
        """
        lam: (B,6)
        t: (T,) time grid; if None, uses torch.linspace(0,1,601) on same device/dtype as lam
        method: torchdiffeq method, e.g. 'rk4', 'dopri5', 'bdf'
        ode_options: dict passed to odeint (e.g., {'rtol':1e-5,'atol':1e-7} for adaptive methods;
                     for rk4 you can pass {'step_size': 1/600} if you want explicit control)

        Returns:
          If return_I=False and return_latent=False:
            U_hat
          Else returns a dict with keys among: 'U_hat', 'I_hat', 'z_t'
        """
        if lam.dim() != 2 or lam.size(1) != self.lam_dim:
            raise ValueError(f"lam must have shape (B, {self.lam_dim}). Got {tuple(lam.shape)}")

        B = lam.size(0)
        device = lam.device
        dtype = lam.dtype

        if t is None:
            t = torch.linspace(0.0, 1.0, 101, device=device, dtype=dtype)
        else:
            t = t.to(device=device, dtype=dtype)

        T = t.numel()

        # Build known IC and encode
        U0 = self.build_initial_state(lam, dtype=dtype)         # (B,3,N)
        z0 = self.encoder(U0, lam)                              # (B,d)

        # Define odeint function signature f(t, z)->dzdt with lam captured
        def f(t_scalar, z):
            return self.dynamics(t_scalar, z, lam)

        # Integrate latent ODE: returns (T, B, d)
        ode_options = ode_options or {}
        # --- torchdiffeq expects rtol/atol as top-level args, NOT inside options ---
        rtol = ode_options.pop("rtol", 1e-4)  # choose your defaults
        atol = ode_options.pop("atol", 1e-6)

        z_t = odeint(
            f, z0, t,
            method=method,
            rtol=rtol,
            atol=atol,
            options=ode_options  # now contains only method-specific options (e.g. step_size)
        )


        # Decode: reshape to (T*B,d) -> (T*B,3,N) -> (B,T,3,N)
        z_flat = z_t.reshape(T * B, -1)
        U_flat = self.decoder(z_flat)                           # (T*B, 3, N)
        U_hat = U_flat.view(T, B, 3, self.N).permute(1, 0, 2, 3).contiguous()

        if not (return_I or return_latent):
            return U_hat

        out = {"U_hat": U_hat}

        if return_I:
            out["I_hat"] = self.compute_intensity(U_hat)        # (B,T,N)

        if return_latent:
            out["z_t"] = z_t.permute(1, 0, 2).contiguous()      # (B,T,d)

        return out


In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = Encoder(N=81, latent_dim=8, feature_dim=256, conv_channels=(32,64,128), kernel_size=5,
                  use_groupnorm=True, gn_groups=8, mlp_hidden=(128,256), activation="gelu", dropout=0.0)

dynamics = LatentDynamics(latent_dim=8, lam_dim=6, width=32, depth=2, film_hidden=(16,),
                          activation="gelu", dropout=0.0, use_layernorm=True, autonomous=True)

decoder = Decoder(N=81, latent_dim=8, base_channels=64, base_length=9,
                  up_lengths=(27,81), conv_channels=(64,32), kernel_size=5,
                  mlp_hidden=(128,256), activation="gelu", dropout=0.0, use_groupnorm=True, gn_groups=8)

model = LatentODESurrogate(encoder, dynamics, decoder, N=81, phi=1.0, r0_index=0).to(device)

lam = torch.randn(10, 6, device=device)  # example
out = model(lam, method="dopri5", return_I=True, return_latent=False)
U_hat = out["U_hat"]   # (B,601,3,81)
I_hat = out["I_hat"]   # (B,601,81)


# The loss function

In [18]:
def compute_losses(
    model,                      # LatentODESurrogate
    U_true: torch.Tensor,       # (B,T,3,N)
    lam: torch.Tensor,          # (B,6)
    I_true: torch.Tensor = None,# (B,T,N) or None
    alpha: float = 1.0,
    beta: float = 1.0,
    gamma: float = 1.0,
    delta: float = 1.0,
    method: str = "dopri5",
    ode_options: dict = None,
    W: torch.Tensor = None,     # optional weights for channels (3,) or (1,1,3,1)
):
    """
    Returns:
      total_loss, losses_dict
    """
    if U_true.dim() != 4:
        raise ValueError(f"U_true must have shape (B,T,3,N). Got {tuple(U_true.shape)}")
    B, T, C, N = U_true.shape
    if C != 3:
        raise ValueError(f"Expected 3 channels (h,c,f). Got C={C}")
    if lam.shape[0] != B:
        raise ValueError(f"Batch mismatch: lam has B={lam.shape[0]}, U_true has B={B}")

    ode_options = ode_options or {}

    # --- Rollout prediction ---
    out = model(lam, t=None, method=method, return_I=(I_true is not None), return_latent=True, ode_options=ode_options)
    U_hat = out["U_hat"]        # (B,T,3,N)
    z_roll = out["z_t"]         # (B,T,d)
    I_hat = out.get("I_hat", None)

    # --- Channel weighting for AE/state losses ---
    # W can be (3,) or broadcastable to (B,T,3,N)
    if W is None:
        W_b = None
    else:
        if W.dim() == 1 and W.numel() == 3:
            W_b = W.view(1, 1, 3, 1).to(U_true.device, U_true.dtype)
        else:
            W_b = W.to(U_true.device, U_true.dtype)

    def weighted_mse(a, b):
        if W_b is None:
            return F.mse_loss(a, b)
        return torch.mean(W_b * (a - b) ** 2)

    # --- L_state: rollout decoded state vs truth ---
    L_state = weighted_mse(U_hat, U_true)

    # --- L_I: intensity loss (if provided) ---
    if I_true is not None:
        if I_true.shape != (B, T, N):
            raise ValueError(f"I_true must have shape (B,T,N)={ (B,T,N) }. Got {tuple(I_true.shape)}")
        L_I = F.mse_loss(I_hat, I_true)
    else:
        L_I = torch.zeros((), device=U_true.device, dtype=U_true.dtype)

    # --- L_AE: snapshot AE reconstruction (no ODE) ---
    # Compute z_enc_k = E(U_k, lam), then U_rec_k = D(z_enc_k)
    # Flatten time into batch for efficiency
    U_flat = U_true.reshape(B * T, 3, N)
    lam_rep = lam[:, None, :].expand(B, T, lam.size(1)).reshape(B * T, lam.size(1))

    z_enc_flat = model.encoder(U_flat, lam_rep)            # (B*T, d)
    U_rec_flat = model.decoder(z_enc_flat)                # (B*T, 3, N)
    U_rec = U_rec_flat.view(B, T, 3, N)

    L_AE = weighted_mse(U_rec, U_true)

    # --- L_dyn: latent rollout vs latent encodings of truth ---
    z_enc = z_enc_flat.view(B, T, -1)                     # (B,T,d)
    L_dyn = F.mse_loss(z_roll, z_enc)

    # --- total ---
    total = alpha * L_AE + beta * L_dyn + gamma * L_state + delta * L_I

    losses = {
        "total": total,
        "L_AE": L_AE,
        "L_dyn": L_dyn,
        "L_state": L_state,
        "L_I": L_I,
    }
    return total, losses


# Model training

In [19]:
# --- instantiate your components (adjust args to what you already chose) ---
encoder  = Encoder(N=81, lam_dim=6, latent_dim=16, feature_dim=256).to(device)
dynamics = LatentDynamics(latent_dim=16, lam_dim=6, width=128, depth=4, autonomous=True).to(device)
decoder  = Decoder(N=81, latent_dim=16).to(device)

model = LatentODESurrogate(
    encoder=encoder,
    dynamics=dynamics,
    decoder=decoder,
    N=81,
    lam_dim=6,
    phi=1.0,          # set your phi here
    r0_index=0        # set index corresponding to r=0 in your grid
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [20]:
# Loss weights
alpha = 1.0
beta  = 1.0
gamma = 1.0
delta = 1.0

# ODE solver choice (you can change to 'rk4' or 'implicit_adams')
ode_method = "dopri5"
ode_options = {"rtol": 1e-4, "atol": 1e-6}

def reduced_loss_hc_and_I(model, I_batch, p_batch, y_batch):
    """
    Used when y_batch has only 2 channels (h,c).
    Trains using:
      - L_state on (h,c) only
      - L_I on intensity
    """
    # y_batch: (B,2,601,81) -> U_true_hc: (B,601,2,81)
    U_true_hc = y_batch.permute(0, 2, 1, 3).contiguous()

    # I_batch: (B,1,601,81) -> I_true: (B,601,81)
    I_true = I_batch.squeeze(1).contiguous()

    out = model(p_batch, method=ode_method, ode_options=ode_options, return_I=True, return_latent=False)
    U_hat = out["U_hat"]  # (B,601,3,81)
    I_hat = out["I_hat"]  # (B,601,81)

    # compare only h,c channels
    L_state_hc = torch.mean((U_hat[:, :, :2, :] - U_true_hc) ** 2)
    L_I = torch.mean((I_hat - I_true) ** 2)

    total = gamma * L_state_hc + delta * L_I
    return total, {"total": total, "L_state_hc": L_state_hc, "L_I": L_I}



In [21]:
start_time = time()

for epoch in range(2):
    # --------------------
    # TRAIN
    # --------------------
    model.train()
    train_loss_sum = 0.0
    n_train = 0

    for (I_batch, p_batch), y_batch in train_loader:
        I_batch = I_batch.to(device, non_blocking=True)   # (B,1,601,81)
        p_batch = p_batch.to(device, non_blocking=True)   # (B,6)
        y_batch = y_batch.to(device, non_blocking=True)   # (B,C,601,81)

        optimizer.zero_grad(set_to_none=True)

        C = y_batch.size(1)
        if C == 3:
            # Full loss (requires y_batch=(h,c,f))
            U_true = y_batch.permute(0, 2, 1, 3).contiguous()          # (B,601,3,81)
            I_true = I_batch.squeeze(1).contiguous()                   # (B,601,81)

            total, losses = compute_losses(
                model=model,
                U_true=U_true,
                lam=p_batch,
                I_true=I_true,
                alpha=alpha, beta=beta, gamma=gamma, delta=delta,
                method=ode_method,
                ode_options=ode_options,
                W=None
            )
        elif C == 2:
            # Reduced loss (your current dataset setup)
            total, losses = reduced_loss_hc_and_I(model, I_batch, p_batch, y_batch)
        else:
            raise ValueError(f"Expected y_batch channels C in {{2,3}}. Got C={C}")

        total.backward()
        optimizer.step()

        bs = p_batch.size(0)
        train_loss_sum += total.item() * bs
        n_train += bs

    train_loss = train_loss_sum / max(1, n_train)

    # --------------------
    # EVAL
    # --------------------
    model.eval()
    val_loss_sum = 0.0
    n_val = 0

    with torch.no_grad():
        for (I_batch, p_batch), y_batch in test_loader:
            I_batch = I_batch.to(device, non_blocking=True)
            p_batch = p_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)

            C = y_batch.size(1)
            if C == 3:
                U_true = y_batch.permute(0, 2, 1, 3).contiguous()
                I_true = I_batch.squeeze(1).contiguous()

                total, _ = compute_losses(
                    model=model,
                    U_true=U_true,
                    lam=p_batch,
                    I_true=I_true,
                    alpha=alpha, beta=beta, gamma=gamma, delta=delta,
                    method=ode_method,
                    ode_options=ode_options,
                    W=None
                )
            elif C == 2:
                total, _ = reduced_loss_hc_and_I(model, I_batch, p_batch, y_batch)
            else:
                raise ValueError(f"Expected y_batch channels C in {{2,3}}. Got C={C}")

            bs = p_batch.size(0)
            val_loss_sum += total.item() * bs
            n_val += bs

    val_loss = val_loss_sum / max(1, n_val)
    print(f"Epoch {epoch}: train_loss = {train_loss:.4e} | val_loss = {val_loss:.4e}")

end_time = time()
print(f"Training with Dropi5 method completed in {end_time - start_time:.4f} seconds.")

c:\Users\arnab\anaconda3\envs\deepxde\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch 0: train_loss = 2.9783e+00 | val_loss = 2.9058e+00
Epoch 1: train_loss = 2.4248e+00 | val_loss = 2.6348e+00
Training with Dropi5 method completed in 17.1823 seconds.


In [22]:
import gc

# Delete large arrays that are no longer needed to free memory
del c_all
del c_train
del c_test
del X_train
del explained_var_ratio
del y_batch
del y0
del out
gc.collect()

20

# Experiment 1: Fastest solver

In [23]:
# Loss weights
alpha = 1.0
beta  = 1.0
gamma = 1.0
delta = 1.0

# Global solver settings (will be overwritten inside run_one_experiment)
ode_method = "dopri5"
ode_options = {"rtol": 1e-4, "atol": 1e-6}

def reduced_loss_hc_and_I(model, I_batch, p_batch, y_batch):
    """
    Used when y_batch has only 2 channels (h,c).
    Trains using:
      - L_state on (h,c) only
      - L_I on intensity
    """
    # y_batch: (B,2,T,N) -> U_true_hc: (B,T,2,N)
    U_true_hc = y_batch.permute(0, 2, 1, 3).contiguous()

    # I_batch: (B,1,T,N) -> I_true: (B,T,N)
    I_true = I_batch.squeeze(1).contiguous()

    out = model(p_batch, method=ode_method, ode_options=ode_options,
                return_I=True, return_latent=False)
    U_hat = out["U_hat"]  # (B,T,3,N)
    I_hat = out["I_hat"]  # (B,T,N)

    # compare only h,c channels
    L_state_hc = torch.mean((U_hat[:, :, :2, :] - U_true_hc) ** 2)
    L_I = torch.mean((I_hat - I_true) ** 2)

    total = gamma * L_state_hc + delta * L_I
    return total, {"total": total, "L_state_hc": L_state_hc, "L_I": L_I}


def run_one_experiment(method_name, method_options, num_epochs=1):
    """
    Run the existing train+eval loop for a given ODE solver method and options,
    and measure wall-clock time.
    """
    global ode_method, ode_options
    ode_method = method_name
    ode_options = dict(method_options)  # make a copy so we can safely modify inside the model

    print(f"\n==== Running experiment with method = {ode_method} ====")
    print(f"ode_options = {ode_options}")
    start_time = time()

    for epoch in range(num_epochs):
        # --------------------
        # TRAIN
        # --------------------
        # --- instantiate your components (adjust args to what you already chose) ---
        encoder  = Encoder(N=81, lam_dim=6, latent_dim=16, feature_dim=256).to(device)
        dynamics = LatentDynamics(latent_dim=16, lam_dim=6, width=128, depth=4, autonomous=True).to(device)
        decoder  = Decoder(N=81, latent_dim=16).to(device)

        model = LatentODESurrogate(
            encoder=encoder,
            dynamics=dynamics,
            decoder=decoder,
            N=81,
            lam_dim=6,
            phi=1.0,          # set your phi here
            r0_index=0        # set index corresponding to r=0 in your grid
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

        model.train()
        train_loss_sum = 0.0
        n_train = 0

        for (I_batch, p_batch), y_batch in train_loader:
            I_batch = I_batch.to(device, non_blocking=True)   # (B,1,T,N)
            p_batch = p_batch.to(device, non_blocking=True)   # (B,6)
            y_batch = y_batch.to(device, non_blocking=True)   # (B,C,T,N)

            optimizer.zero_grad(set_to_none=True)

            C = y_batch.size(1)
            if C == 3:
                # Full loss (requires y_batch=(h,c,f))
                U_true = y_batch.permute(0, 2, 1, 3).contiguous()          # (B,T,3,N)
                I_true = I_batch.squeeze(1).contiguous()                   # (B,T,N)

                total, losses = compute_losses(
                    model=model,
                    U_true=U_true,
                    lam=p_batch,
                    I_true=I_true,
                    alpha=alpha, beta=beta, gamma=gamma, delta=delta,
                    method=ode_method,
                    ode_options=ode_options,
                    W=None
                )
            elif C == 2:
                # Reduced loss (current dataset setup: only h,c supervised)
                total, losses = reduced_loss_hc_and_I(model, I_batch, p_batch, y_batch)
            else:
                raise ValueError(f"Expected y_batch channels C in {{2,3}}. Got C={C}")

            total.backward()
            optimizer.step()

            bs = p_batch.size(0)
            train_loss_sum += total.item() * bs
            n_train += bs

        train_loss = train_loss_sum / max(1, n_train)

        # --------------------
        # EVAL
        # --------------------
        model.eval()
        val_loss_sum = 0.0
        n_val = 0

        with torch.no_grad():
            for (I_batch, p_batch), y_batch in test_loader:
                I_batch = I_batch.to(device, non_blocking=True)
                p_batch = p_batch.to(device, non_blocking=True)
                y_batch = y_batch.to(device, non_blocking=True)

                C = y_batch.size(1)
                if C == 3:
                    U_true = y_batch.permute(0, 2, 1, 3).contiguous()
                    I_true = I_batch.squeeze(1).contiguous()

                    total, _ = compute_losses(
                        model=model,
                        U_true=U_true,
                        lam=p_batch,
                        I_true=I_true,
                        alpha=alpha, beta=beta, gamma=gamma, delta=delta,
                        method=ode_method,
                        ode_options=ode_options,
                        W=None
                    )
                elif C == 2:
                    total, _ = reduced_loss_hc_and_I(model, I_batch, p_batch, y_batch)
                else:
                    raise ValueError(f"Expected y_batch channels C in {{2,3}}. Got C={C}")

                bs = p_batch.size(0)
                val_loss_sum += total.item() * bs
                n_val += bs

        val_loss = val_loss_sum / max(1, n_val)
        print(f"Epoch {epoch}: train_loss = {train_loss:.4e} | val_loss = {val_loss:.4e}")

    elapsed = time() - start_time
    print(f"Total wall-clock time with method={ode_method}: {elapsed:.2f} s")
    return elapsed


# -------------------------------------------------
# Run the speed experiment for dopri5, rk4, and implicit_adams
# -------------------------------------------------

times = {}

# 1) dopri5 (adaptive explicit) – your current setup
times["dopri5"] = run_one_experiment(
    method_name="dopri5",
    method_options={"rtol": 1e-4, "atol": 1e-6},
    num_epochs=1,        # increase if you want more robust timing
)

# 2) rk4 (fixed-step explicit) – specify step_size ~ 1/100 for T=101
times["rk4"] = run_one_experiment(
    method_name="rk4",
    method_options={"step_size": 1.0 / 100.0},
    num_epochs=1,
)

# 3) implicit_adams (stiff solver) – adaptive implicit
times["implicit_adams"] = run_one_experiment(
    method_name="implicit_adams",
    method_options={"rtol": 1e-3, "atol": 1e-5},
    num_epochs=1,
)

print("\nSummary of timings (seconds):")
for m, t_sec in times.items():
    print(f"  {m:6s}: {t_sec:.2f} s")



==== Running experiment with method = dopri5 ====
ode_options = {'rtol': 0.0001, 'atol': 1e-06}
Epoch 0: train_loss = 2.7927e+00 | val_loss = 2.8159e+00
Total wall-clock time with method=dopri5: 7.62 s

==== Running experiment with method = rk4 ====
ode_options = {'step_size': 0.01}
Epoch 0: train_loss = 3.1761e+00 | val_loss = 2.9079e+00
Total wall-clock time with method=rk4: 14.47 s

==== Running experiment with method = implicit_adams ====
ode_options = {'rtol': 0.001, 'atol': 1e-05}
Epoch 0: train_loss = 3.4039e+00 | val_loss = 2.9071e+00
Total wall-clock time with method=implicit_adams: 11.50 s

Summary of timings (seconds):
  dopri5: 7.62 s
  rk4   : 14.47 s
  implicit_adams: 11.50 s


In [24]:
# # Loss weights
# alpha = 1.0
# beta  = 1.0
# gamma = 1.0
# delta = 1.0

# # Global solver settings (will be overwritten inside run_one_experiment)
# ode_method = "dopri5"
# ode_options = {"rtol": 1e-4, "atol": 1e-6}

def reduced_loss_hc_and_I(model, I_batch, p_batch, y_batch):
    """
    Used when y_batch has only 2 channels (h,c).
    Trains using:
      - L_state on (h,c) only
      - L_I on intensity
    """
    # y_batch: (B,2,T,N) -> U_true_hc: (B,T,2,N)
    U_true_hc = y_batch.permute(0, 2, 1, 3).contiguous()

    # I_batch: (B,1,T,N) -> I_true: (B,T,N)
    I_true = I_batch.squeeze(1).contiguous()

    out = model(p_batch, method=ode_method, ode_options=ode_options,
                return_I=True, return_latent=False)
    U_hat = out["U_hat"]  # (B,T,3,N)
    I_hat = out["I_hat"]  # (B,T,N)

    # compare only h,c channels
    L_state_hc = torch.mean((U_hat[:, :, :2, :] - U_true_hc) ** 2)
    L_I = torch.mean((I_hat - I_true) ** 2)

    total = gamma * L_state_hc + delta * L_I
    return total, {"total": total, "L_state_hc": L_state_hc, "L_I": L_I}


def run_one_experiment(method_name, method_options, num_epochs=1):
    """
    Run the existing train+eval loop for a given ODE solver method and options,
    and measure wall-clock time.
    """
    global ode_method, ode_options
    ode_method = method_name
    ode_options = dict(method_options)  # make a copy so we can safely modify inside the model

    print(f"\n==== Running experiment with method = {ode_method} ====")
    print(f"ode_options = {ode_options}")
    start_time = time()

    for epoch in range(num_epochs):
        # --------------------
        # TRAIN
        # --------------------
        # --- instantiate your components (adjust args to what you already chose) ---
        encoder  = Encoder(N=81, lam_dim=6, latent_dim=16, feature_dim=256).to(device)
        dynamics = LatentDynamics(latent_dim=16, lam_dim=6, width=128, depth=4, autonomous=True).to(device)
        decoder  = Decoder(N=81, latent_dim=16).to(device)

        model = LatentODESurrogate(
            encoder=encoder,
            dynamics=dynamics,
            decoder=decoder,
            N=81,
            lam_dim=6,
            phi=1.0,          # set your phi here
            r0_index=0        # set index corresponding to r=0 in your grid
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

        model.train()
        train_loss_sum = 0.0
        n_train = 0

        for (I_batch, p_batch), y_batch in train_loader:
            I_batch = I_batch.to(device, non_blocking=True)   # (B,1,T,N)
            p_batch = p_batch.to(device, non_blocking=True)   # (B,6)
            y_batch = y_batch.to(device, non_blocking=True)   # (B,C,T,N)

            optimizer.zero_grad(set_to_none=True)

            C = y_batch.size(1)
            if C == 3:
                # Full loss (requires y_batch=(h,c,f))
                U_true = y_batch.permute(0, 2, 1, 3).contiguous()          # (B,T,3,N)
                I_true = I_batch.squeeze(1).contiguous()                   # (B,T,N)

                total, losses = compute_losses(
                    model=model,
                    U_true=U_true,
                    lam=p_batch,
                    I_true=I_true,
                    alpha=alpha, beta=beta, gamma=gamma, delta=delta,
                    method=ode_method,
                    ode_options=ode_options,
                    W=None
                )
            elif C == 2:
                # Reduced loss (current dataset setup: only h,c supervised)
                total, losses = reduced_loss_hc_and_I(model, I_batch, p_batch, y_batch)
            else:
                raise ValueError(f"Expected y_batch channels C in {{2,3}}. Got C={C}")

            total.backward()
            optimizer.step()

            bs = p_batch.size(0)
            train_loss_sum += total.item() * bs
            n_train += bs

        train_loss = train_loss_sum / max(1, n_train)

        # --------------------
        # EVAL
        # --------------------
        model.eval()
        val_loss_sum = 0.0
        n_val = 0

        with torch.no_grad():
            for (I_batch, p_batch), y_batch in test_loader:
                I_batch = I_batch.to(device, non_blocking=True)
                p_batch = p_batch.to(device, non_blocking=True)
                y_batch = y_batch.to(device, non_blocking=True)

                C = y_batch.size(1)
                if C == 3:
                    U_true = y_batch.permute(0, 2, 1, 3).contiguous()
                    I_true = I_batch.squeeze(1).contiguous()

                    total, _ = compute_losses(
                        model=model,
                        U_true=U_true,
                        lam=p_batch,
                        I_true=I_true,
                        alpha=alpha, beta=beta, gamma=gamma, delta=delta,
                        method=ode_method,
                        ode_options=ode_options,
                        W=None
                    )
                elif C == 2:
                    total, _ = reduced_loss_hc_and_I(model, I_batch, p_batch, y_batch)
                else:
                    raise ValueError(f"Expected y_batch channels C in {{2,3}}. Got C={C}")

                bs = p_batch.size(0)
                val_loss_sum += total.item() * bs
                n_val += bs

        val_loss = val_loss_sum / max(1, n_val)
        print(f"Epoch {epoch}: train_loss = {train_loss:.4e} | val_loss = {val_loss:.4e}")

    elapsed = time() - start_time
    print(f"Total wall-clock time with method={ode_method}: {elapsed:.2f} s")
    return elapsed


# -------------------------------------------------
# Run the speed experiment for dopri5, rk4, and implicit_adams
# -------------------------------------------------

times = {}

# 1) dopri8
times["dopri8"] = run_one_experiment(
    method_name="dopri8",
    method_options={"rtol": 1e-4, "atol": 1e-6},
    num_epochs=1,        # increase if you want more robust timing
)

# 4) bosh3 (Bogacki–Shampine 3(2), adaptive explicit)
times["bosh3"] = run_one_experiment(
    method_name="bosh3",
    method_options={"rtol": 1e-3, "atol": 1e-5},  # slightly looser than dopri5
    num_epochs=1,
)

# 5) adaptive_heun (2nd-order adaptive explicit)
times["adaptive_heun"] = run_one_experiment(
    method_name="adaptive_heun",
    method_options={"rtol": 1e-3, "atol": 1e-5},
    num_epochs=1,
)

#    Here step_size is coarse; you can try 1/50, 1/100, 1/200 to see speed vs accuracy.
times["midpoint"] = run_one_experiment(
    method_name="midpoint",
    method_options={"step_size": 1.0 / 100.0},
    num_epochs=1,
)


print("\nSummary of timings (seconds):")
for m, t_sec in times.items():
    print(f"  {m:6s}: {t_sec:.2f} s")



==== Running experiment with method = dopri8 ====
ode_options = {'rtol': 0.0001, 'atol': 1e-06}
Epoch 0: train_loss = 2.6880e+00 | val_loss = 2.7863e+00
Total wall-clock time with method=dopri8: 7.90 s

==== Running experiment with method = bosh3 ====
ode_options = {'rtol': 0.001, 'atol': 1e-05}
Epoch 0: train_loss = 2.8586e+00 | val_loss = 2.7312e+00
Total wall-clock time with method=bosh3: 7.45 s

==== Running experiment with method = adaptive_heun ====
ode_options = {'rtol': 0.001, 'atol': 1e-05}
Epoch 0: train_loss = 3.0691e+00 | val_loss = 3.0715e+00
Total wall-clock time with method=adaptive_heun: 8.84 s

==== Running experiment with method = midpoint ====
ode_options = {'step_size': 0.01}
Epoch 0: train_loss = 3.2082e+00 | val_loss = 3.2366e+00
Total wall-clock time with method=midpoint: 9.91 s

Summary of timings (seconds):
  dopri8: 7.90 s
  bosh3 : 7.45 s
  adaptive_heun: 8.84 s
  midpoint: 9.91 s


dropi5 turned out to be the fastest solver among the ones tested. It consistently achieved lower training times across multiple runs while maintaining one of the best accuracy compared to the other solvers.

# Experiment 2: best choice for rtol and atol for dropi5

In [25]:
# Loss weights
alpha = 1.0
beta  = 1.0
gamma = 1.0
delta = 1.0


# Global solver settings (will be overwritten inside run_one_experiment)
ode_method = "dopri5"
ode_options = {"rtol": 1e-4, "atol": 1e-6}

def reduced_loss_hc_and_I(model, I_batch, p_batch, y_batch):
    """
    Used when y_batch has only 2 channels (h,c).
    Trains using:
      - L_state on (h,c) only
      - L_I on intensity
    """
    # y_batch: (B,2,T,N) -> U_true_hc: (B,T,2,N)
    U_true_hc = y_batch.permute(0, 2, 1, 3).contiguous()

    # I_batch: (B,1,T,N) -> I_true: (B,T,N)
    I_true = I_batch.squeeze(1).contiguous()

    out = model(p_batch, method=ode_method, ode_options=ode_options,
                return_I=True, return_latent=False)
    U_hat = out["U_hat"]  # (B,T,3,N)
    I_hat = out["I_hat"]  # (B,T,N)

    # compare only h,c channels
    L_state_hc = torch.mean((U_hat[:, :, :2, :] - U_true_hc) ** 2)
    L_I = torch.mean((I_hat - I_true) ** 2)

    total = gamma * L_state_hc + delta * L_I
    return total, {"total": total, "L_state_hc": L_state_hc, "L_I": L_I}


def run_one_experiment(method_name, method_options, num_epochs=1):
    """
    Run the existing train+eval loop for a given ODE solver method and options,
    and measure wall-clock time.
    """
    global ode_method, ode_options
    ode_method = method_name
    ode_options = dict(method_options)  # make a copy so we can safely modify inside the model

    print(f"\n==== Running experiment with method = {ode_method} ====")
    print(f"ode_options = {ode_options}")
    start_time = time()

    for epoch in range(num_epochs):
        # --------------------
        # TRAIN
        # --------------------
        # --- instantiate your components (adjust args to what you already chose) ---
        encoder  = Encoder(N=81, lam_dim=6, latent_dim=16, feature_dim=256).to(device)
        dynamics = LatentDynamics(latent_dim=16, lam_dim=6, width=128, depth=4, autonomous=True).to(device)
        decoder  = Decoder(N=81, latent_dim=16).to(device)

        model = LatentODESurrogate(
            encoder=encoder,
            dynamics=dynamics,
            decoder=decoder,
            N=81,
            lam_dim=6,
            phi=1.0,          # set your phi here
            r0_index=0        # set index corresponding to r=0 in your grid
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

        model.train()
        train_loss_sum = 0.0
        n_train = 0

        for (I_batch, p_batch), y_batch in train_loader:
            I_batch = I_batch.to(device, non_blocking=True)   # (B,1,T,N)
            p_batch = p_batch.to(device, non_blocking=True)   # (B,6)
            y_batch = y_batch.to(device, non_blocking=True)   # (B,C,T,N)

            optimizer.zero_grad(set_to_none=True)

            C = y_batch.size(1)
            if C == 3:
                # Full loss (requires y_batch=(h,c,f))
                U_true = y_batch.permute(0, 2, 1, 3).contiguous()          # (B,T,3,N)
                I_true = I_batch.squeeze(1).contiguous()                   # (B,T,N)

                total, losses = compute_losses(
                    model=model,
                    U_true=U_true,
                    lam=p_batch,
                    I_true=I_true,
                    alpha=alpha, beta=beta, gamma=gamma, delta=delta,
                    method=ode_method,
                    ode_options=ode_options,
                    W=None
                )
            elif C == 2:
                # Reduced loss (current dataset setup: only h,c supervised)
                total, losses = reduced_loss_hc_and_I(model, I_batch, p_batch, y_batch)
            else:
                raise ValueError(f"Expected y_batch channels C in {{2,3}}. Got C={C}")

            total.backward()
            optimizer.step()

            bs = p_batch.size(0)
            train_loss_sum += total.item() * bs
            n_train += bs

        train_loss = train_loss_sum / max(1, n_train)

        # --------------------
        # EVAL
        # --------------------
        model.eval()
        val_loss_sum = 0.0
        n_val = 0

        with torch.no_grad():
            for (I_batch, p_batch), y_batch in test_loader:
                I_batch = I_batch.to(device, non_blocking=True)
                p_batch = p_batch.to(device, non_blocking=True)
                y_batch = y_batch.to(device, non_blocking=True)

                C = y_batch.size(1)
                if C == 3:
                    U_true = y_batch.permute(0, 2, 1, 3).contiguous()
                    I_true = I_batch.squeeze(1).contiguous()

                    total, _ = compute_losses(
                        model=model,
                        U_true=U_true,
                        lam=p_batch,
                        I_true=I_true,
                        alpha=alpha, beta=beta, gamma=gamma, delta=delta,
                        method=ode_method,
                        ode_options=ode_options,
                        W=None
                    )
                elif C == 2:
                    total, _ = reduced_loss_hc_and_I(model, I_batch, p_batch, y_batch)
                else:
                    raise ValueError(f"Expected y_batch channels C in {{2,3}}. Got C={C}")

                bs = p_batch.size(0)
                val_loss_sum += total.item() * bs
                n_val += bs

        val_loss = val_loss_sum / max(1, n_val)
        print(f"Epoch {epoch}: train_loss = {train_loss:.4e} | val_loss = {val_loss:.4e}")

    elapsed = time() - start_time
    print(f"Total wall-clock time with method={ode_method}: {elapsed:.2f} s")
    return elapsed, val_loss

In [26]:
# Grid of (rtol, atol) settings for dopri5
dopri5_configs = {
    "loose1":  {"rtol": 3e-3, "atol": 1e-5},
    "loose2":  {"rtol": 1e-3, "atol": 1e-5},
    "medium":  {"rtol": 3e-4, "atol": 1e-6},
    "tight":   {"rtol": 1e-4, "atol": 1e-6},   # your current choice
}

dopri5_times = {}
dopri5_val_losses = {}

# IMPORTANT:
# This will continue training the SAME model across configs.
# For a quick *speed vs loss* comparison on a 1% subset, that's fine.
# For a perfectly fair comparison, you would re-init model+optimizer each time.

for name, opts in dopri5_configs.items():
    print(f"\n=== dopri5 config: {name} | opts = {opts} ===")
    elapsed, val_loss = run_one_experiment(
        method_name="dopri5",
        method_options=opts,
        num_epochs=1,        # 1 epoch is enough for speed comparison on a subset
    )
    dopri5_times[name] = elapsed
    dopri5_val_losses[name] = val_loss

print("\nSummary for dopri5 (1% data, 1 epoch each):")
for name in dopri5_configs.keys():
    print(f"  {name:7s} | time = {dopri5_times[name]:6.2f} s | val_loss = {dopri5_val_losses[name]:.4e}")



=== dopri5 config: loose1 | opts = {'rtol': 0.003, 'atol': 1e-05} ===

==== Running experiment with method = dopri5 ====
ode_options = {'rtol': 0.003, 'atol': 1e-05}
Epoch 0: train_loss = 3.4977e+00 | val_loss = 3.5498e+00
Total wall-clock time with method=dopri5: 6.55 s

=== dopri5 config: loose2 | opts = {'rtol': 0.001, 'atol': 1e-05} ===

==== Running experiment with method = dopri5 ====
ode_options = {'rtol': 0.001, 'atol': 1e-05}
Epoch 0: train_loss = 2.6661e+00 | val_loss = 2.6972e+00
Total wall-clock time with method=dopri5: 6.65 s

=== dopri5 config: medium | opts = {'rtol': 0.0003, 'atol': 1e-06} ===

==== Running experiment with method = dopri5 ====
ode_options = {'rtol': 0.0003, 'atol': 1e-06}
Epoch 0: train_loss = 2.9022e+00 | val_loss = 2.8865e+00
Total wall-clock time with method=dopri5: 6.46 s

=== dopri5 config: tight | opts = {'rtol': 0.0001, 'atol': 1e-06} ===

==== Running experiment with method = dopri5 ====
ode_options = {'rtol': 0.0001, 'atol': 1e-06}
Epoch 0: tr

loose2 was found to be the best choice for rtol and atol for the dropi5 solver. This configuration provided not only fast convergence but also maintained high accuracy, making it the optimal setting for this solver.

# Experiment 3: best choice for alpha, beta, gamma and delta

In [27]:
with torch.no_grad():
    (I_batch, p_batch), y_batch = next(iter(train_loader))

    I_batch = I_batch.to(device)
    p_batch = p_batch.to(device)
    y_batch = y_batch.to(device)

    U_true = y_batch.permute(0, 2, 1, 3).contiguous()  # (B,T,3,N)
    I_true = I_batch.squeeze(1).contiguous()           # (B,T,N)

    total, losses = compute_losses(
        model=model,
        U_true=U_true,
        lam=p_batch,
        I_true=I_true,
        alpha=1.0, beta=1.0, gamma=1.0, delta=1.0,
        method="dopri5",
        ode_options={"rtol": 1e-4, "atol": 1e-6},
        W=None
    )

print("L_AE    =", losses["L_AE"].item())
print("L_dyn   =", losses["L_dyn"].item())
print("L_state =", losses["L_state"].item())
print("L_I     =", losses["L_I"].item())


L_AE    = 1.2392044067382812
L_dyn   = 0.01545071229338646
L_state = 1.2401463985443115
L_I     = 0.039498355239629745


In [29]:
eps = 1e-8
m_AE    = losses["L_AE"].item()
m_dyn   = losses["L_dyn"].item()
m_state = losses["L_state"].item()
m_I     = losses["L_I"].item()

alpha0 = 0.5 / (m_AE    + eps)
beta0  = 0.5 / (m_dyn   + eps)
gamma0 = 1.0 / (m_state + eps)
delta0 = 1.0 / (m_I     + eps)

s = (alpha0 + beta0 + gamma0 + delta0)
alpha = alpha0 / s
beta  = beta0  / s
gamma = gamma0 / s
delta = delta0 / s

print(alpha, beta, gamma, delta)


0.006851696211074891 0.5495310863588104 0.013692983583896857 0.42992423384621786


At first the different losses are scaled to be in the same range dividing by 1/(m+eps). Then they are weighted as per priority or importance. For this experiment, getting accurate prediction for h, c, f and I is more important than getting accurate encoding or consistent dynamics. Therefore, the weights are set as follows: alpha0=0.5/(m_AE+eps), beta0=0.5/(m_dyn+eps), gamma0=1.0/(m_state+eps), delta0=1.0/(m_I+eps). Finally, all weights are normalized to sum to 1.